## Market Daily Data Collection & Processing

Tickers of interest:
* GameStop - GME
* AMC - AMC
* BBBY - Bed Bath & Beyond
* BB - BlackBerry 
* NOK - Nokia Oyj


In [1]:
import os
import requests 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt
import seaborn as sns

## Import data 
* Alpha Vantage free plan does not work so we employed yfinance instead. 

In [3]:
import yfinance as yf

df = yf.download(["GME", "AMC", "BBBY", "BB", "NOK"],
                 start="2019-01-01",
                 end="2023-06-30",
                 auto_adjust=True)

[*********************100%***********************]  5 of 5 completed

1 Failed download:
['BBBY']: OperationalError('database is locked')


In [6]:
# Redirect cache to a throwaway temp location
import tempfile, os
yf.set_tz_cache_location(os.path.join(tempfile.gettempdir(), "yf_temp"))

df = yf.download(["GME", "AMC", "BBBY", "BB", "NOK"],
                 start="2019-01-01",
                 end="2023-06-30",
                 auto_adjust=True)


[*********************100%***********************]  5 of 5 completed


In [12]:
# examine data format

df.head()

Price            Close                                         High        \
Ticker             AMC    BB   BBBY       GME       NOK         AMC    BB   
Date                                                                        
2019-01-02  119.143227  7.11  14.13  3.160276  5.049535  120.899419  7.20   
2019-01-03  120.714569  6.88  13.25  3.136097  4.899983  125.151234  7.06   
2019-01-04  125.151237  7.23  14.26  3.684974  5.216679  126.537699  7.25   
2019-01-07  130.512207  7.43  15.28  3.743005  5.295853  131.806246  7.46   
2019-01-08  134.671600  7.41  15.76  3.822798  5.410214  135.873203  7.60   

Price                                      ...        Open               \
Ticker           BBBY       GME       NOK  ...         AMC    BB   BBBY   
Date                                       ...                            
2019-01-02  14.460000  3.186874  5.075926  ...  111.009325  7.00  13.31   
2019-01-03  14.460000  3.206218  4.970360  ...  118.311364  7.02  13.98   
2019-01-04  14.970000  3.697064  5.243070  ...  122.193456  7.09  13.36   
2019-01-07  15.540000  3.822798  5.357432  ...  125.243657  7.32  14.65   
2019-01-08  16.360001  3.842142  5.419011  ...  131.344102  7.53  15.40   

Price                           Volume                                        
Ticker           GME       NOK     AMC       BB     BBBY       GME       NOK  
Date                                                                          
2019-01-02  3.010363  5.014346  137490  3378700  1238500   8496800  24338600  
2019-01-03  3.138515  4.961563  139240  3692500  1396300   7001600  17354500  
2019-01-04  3.392401  5.040737  102500  3823500  1708300  47662800  34666300  
2019-01-07  3.653540  5.234273  111920  4308700  1457300  18872000  25502600  
2019-01-08  3.747841  5.339837  112270  3960900  1391100  13775200  33497100  

[5 rows x 25 columns]

In [14]:
# --- Reshape from wide MultiIndex to long flat format ---
df_long = df.stack(level='Ticker').reset_index()
df_long.columns.name = None

# Rename to your required schema
df_long = df_long.rename(columns={
    'Date':   'date',
    'Ticker': 'ticker',
    'Open':   'open',
    'High':   'high',
    'Low':    'low',
    'Close':  'close',        # this is already adjusted close with auto_adjust=True
    'Volume': 'volume'
})

# auto_adjust=True means Close IS adjusted close — make that explicit
df_long['adj_close'] = df_long['close']

# Standardize date format
df_long['date'] = pd.to_datetime(df_long['date']).dt.date

# Sort cleanly
df_long = df_long.sort_values(['ticker', 'date']).reset_index(drop=True)

# Keep only the columns your schema requires
df_long = df_long[['date', 'ticker', 'open', 'high', 'low', 'close', 'adj_close', 'volume']]

print(df_long.head(10))
print(f"\nShape: {df_long.shape}")
print(f"\nTickers: {df_long['ticker'].unique()}")
print(f"\nDate range: {df_long['date'].min()} to {df_long['date'].max()}")
print(f"\nMissing values:\n{df_long.isnull().sum()}")


         date ticker        open        high         low       close  \
0  2019-01-02    AMC  111.009325  120.899419  109.992588  119.143227   
1  2019-01-03    AMC  118.311364  125.151234  116.740044  120.714569   
2  2019-01-04    AMC  122.193456  126.537699  120.344841  125.151237   
3  2019-01-07    AMC  125.243657  131.806246  123.579901  130.512207   
4  2019-01-08    AMC  131.344102  135.873203  130.234922  134.671600   
5  2019-01-09    AMC  136.242927  136.335363  128.386313  128.940903   
6  2019-01-10    AMC  127.554437  128.663604  124.319361  126.907425   
7  2019-01-11    AMC  127.184731  130.142513  125.798270  129.680359   
8  2019-01-14    AMC  129.218199  131.806263  127.554442  130.604660   
9  2019-01-15    AMC  131.066815  133.562442  129.125763  132.268417   

    adj_close  volume  
0  119.143227  137490  
1  120.714569  139240  
2  125.151237  102500  
3  130.512207  111920  
4  134.671600  112270  
5  128.940903  144360  
6  126.907425   97650  
7  129.680359  

## Calculate daily returns

In [16]:
# Calculate daily returns 

df_long['daily_return'] = df_long.groupby('ticker')['adj_close'].pct_change()

df_long.head()

,date,ticker,open,high,low,close,adj_close,volume,daily_return
0,2019-01-02,AMC,111.009325,120.899419,109.992588,119.143227,119.143227,137490,NaN
1,2019-01-03,AMC,118.311364,125.151234,116.740044,120.714569,120.714569,139240,0.013189
2,2019-01-04,AMC,122.193456,126.537699,120.344841,125.151237,125.151237,102500,0.036753
3,2019-01-07,AMC,125.243657,131.806246,123.579901,130.512207,130.512207,111920,0.042836
4,2019-01-08,AMC,131.344102,135.873203,130.234922,134.671600,134.671600,112270,0.031870


## Identify key events: 

1. `January 13, 2021`: GME squeeze begins, stock surges ~50% in a single day. ([source](https://bakercityherald.com/2023/09/01/an-in-depth-timeline-of-the-gamestop-short-squeeze/))

2. `January 26, 2021`: Elon Musk tweets "Gamestonk!!" linking to `r/WallStreetBets`, GME closes at $147.98. ([source](https://bakercityherald.com/2023/09/01/an-in-depth-timeline-of-the-gamestop-short-squeeze/))
3. `January 27, 2021`: GME hits its highest squeeze close at $347.51. US equity trading volume reaches its highest-ever single-day level.([source](https://www.thestreet.com/investing/stocks/a-timeline-of-the-gamestop-short-squeeze))
4. `January 28, 2021`: GME reaches pre-market high of over $500. Robinhood halts buying of GME, AMC, BB, and NOK. ([source](https://en.wikipedia.org/wiki/GameStop_short_squeeze))
5. `January 28, 2021`: AMC surges 300% to $20 as Reddit triggers its short squeeze. ([source](https://www.cnbc.com/2022/01/26/how-amc-rode-the-meme-stock-rally-to-revitalize-its-business.html))
6. `June 2021`: AMC hits its all-time high of $72.62. ([source](https://www.cnbc.com/2022/01/26/how-amc-rode-the-meme-stock-rally-to-revitalize-its-business.html))
7. `April 23, 2023`: BBBY files for Chapter 11 bankruptcy. ([source](https://www.sec.gov/Archives/edgar/data/886158/000119312523172557/d444406d8k.htm))

In [18]:
# Identify key events

events = pd.DataFrame([
    {"date": "2021-01-13", "ticker": "GME",  "event": "GME squeeze begins, +50% single day"},
    {"date": "2021-01-26", "ticker": "GME",  "event": "Elon Musk tweets 'Gamestonk!!'"},
    {"date": "2021-01-27", "ticker": "GME",  "event": "GME peak close ($347). Record US equity volume day"},
    {"date": "2021-01-28", "ticker": "ALL",  "event": "Robinhood halts buying of GME, AMC, BB, NOK"},
    {"date": "2021-01-28", "ticker": "AMC",  "event": "AMC surges 300% to $20"},
    {"date": "2021-06-02", "ticker": "AMC",  "event": "AMC all-time high ($72.62)"},
    {"date": "2022-08-17", "ticker": "BBBY", "event": "BBBY second squeeze peak"},
    {"date": "2023-04-23", "ticker": "BBBY", "event": "BBBY files Chapter 11 bankruptcy"},
])

events["date"] = pd.to_datetime(events["date"])
events.to_csv("data/processed/event_timeline.csv", index=False)


In [19]:
events.head()

,date,ticker,event
0,2021-01-13,GME,"GME squeeze begins, +50% single day"
1,2021-01-26,GME,Elon Musk tweets 'Gamestonk!!'
2,2021-01-27,GME,GME peak close ($347). Record US equity volume...
3,2021-01-28,ALL,"Robinhood halts buying of GME, AMC, BB, NOK"
4,2021-01-28,AMC,AMC surges 300% to $20


## Flag statistically abnormal volume and price days

In [20]:
# --- Flag statistically abnormal volume and price days ---

# Step 1: Calculate rolling mean and std (using full history per ticker)
stats = df_long.groupby('ticker')[['volume', 'daily_return']].transform(
    lambda x: x  # placeholder - to calculate below
)

# Step 2: For each ticker, compute mean and std of volume and daily_return
df_long['volume_mean'] = df_long.groupby('ticker')['volume'].transform('mean')
df_long['volume_std']  = df_long.groupby('ticker')['volume'].transform('std')

df_long['return_mean'] = df_long.groupby('ticker')['daily_return'].transform('mean')
df_long['return_std']  = df_long.groupby('ticker')['daily_return'].transform('std')

# Step 3: Calculate z-scores (how many std deviations away from mean)
df_long['volume_zscore'] = (df_long['volume'] - df_long['volume_mean']) / df_long['volume_std']
df_long['return_zscore'] = (df_long['daily_return'] - df_long['return_mean']) / df_long['return_std']

# Step 4: Flag anything beyond 2 standard deviations
df_long['volume_spike']  = df_long['volume_zscore'].abs() > 2
df_long['return_spike']  = df_long['return_zscore'].abs() > 2

# Step 5: Preview the flagged days
spikes = df_long[df_long['volume_spike'] | df_long['return_spike']].copy()
print(spikes[['date', 'ticker', 'adj_close', 'volume', 'daily_return', 
              'volume_zscore', 'return_zscore', 'volume_spike', 'return_spike']]
      .sort_values(['ticker', 'date']))

            date ticker  adj_close    volume  daily_return  volume_zscore  \
305   2020-03-19    AMC  33.700001    861460      0.364372      -0.397082   
317   2020-04-06    AMC  28.799999    691200      0.268722      -0.418096   
325   2020-04-17    AMC  32.000000   2679990      0.311475      -0.172639   
333   2020-04-29    AMC  51.900002   2716240      0.253623      -0.168165   
341   2020-05-11    AMC  53.200001  10655330      0.297561       0.811678   
...          ...    ...        ...       ...           ...            ...   
5322  2022-03-03    NOK   4.455423  51214000     -0.069418       0.455909   
5418  2022-07-21    NOK   4.618390  37441000      0.089362       0.152577   
5482  2022-10-20    NOK   3.767215  62443200     -0.087719       0.703217   
5497  2022-11-10    NOK   4.221761  27145400      0.059361      -0.074170   
5606  2023-04-20    NOK   3.839278  58372900     -0.090909       0.613574   

      return_zscore  volume_spike  return_spike  
305        2.977242      

In [21]:
# Should show known events clustering in Jan 2021
print(df_long[df_long['volume_spike']]
      .groupby('ticker')['date']
      .apply(lambda x: x.sort_values().head(5)))


ticker      
AMC     515     2021-01-19
        518     2021-01-22
        519     2021-01-25
        520     2021-01-26
        521     2021-01-27
BB      1614    2020-12-01
        1615    2020-12-02
        1645    2021-01-15
        1646    2021-01-19
        1647    2021-01-20
BBBY    2278    2019-01-25
        2380    2019-06-21
        2383    2019-06-26
        2384    2019-06-27
        2390    2019-07-08
GME     3499    2019-06-05
        3812    2020-08-31
        3839    2020-10-08
        3840    2020-10-09
        3844    2020-10-15
NOK     4729    2019-10-24
        4985    2020-10-29
        5043    2021-01-25
        5044    2021-01-26
        5045    2021-01-27
Name: date, dtype: object


* note: the z-score is calculated against the entire history (2019–2023). That means BBBY's squeeze in Aug 2022 is being compared to its own normal baseline

In [23]:
# Confirm the core squeeze days are flagged for all tickers
df_long['date'] = pd.to_datetime(df_long['date'])

squeeze_window = df_long[
    (df_long['date'] >= '2021-01-25') & 
    (df_long['date'] <= '2021-02-05') &
    (df_long['volume_spike'] == True)
][['date', 'ticker', 'volume', 'volume_zscore']].sort_values(['date', 'ticker'])

print(squeeze_window)


           date ticker      volume  volume_zscore
519  2021-01-25    AMC    44323810       4.967042
1650 2021-01-25     BB   363829100      11.234253
3912 2021-01-25    GME   711496000      11.429071
5043 2021-01-25    NOK   296379300       5.855355
520  2021-01-26    AMC    45685020       5.135043
1651 2021-01-26     BB   242740800       7.375827
2782 2021-01-26   BBBY    13347700       3.908819
3913 2021-01-26    GME   714352000      11.476734
5044 2021-01-26    NOK   379387300       7.683497
521  2021-01-27    AMC   122234250      14.582751
1652 2021-01-27     BB   372222600      11.501708
3914 2021-01-27    GME   373586800       5.789786
5045 2021-01-27    NOK  1123003300      24.060669
522  2021-01-28    AMC    59122390       6.793483
1653 2021-01-28     BB   194906600       5.851612
3915 2021-01-28    GME   235263200       3.481336
5046 2021-01-28    NOK   675220900      14.198846
523  2021-01-29    AMC    60219330       6.928868
1654 2021-01-29     BB    94049200       2.637835


## Calculate abnormal returns


In [24]:
# Step 1: Download S&P 500 as your market benchmark
spy = yf.download("^GSPC", start="2019-01-01", end="2023-06-30", auto_adjust=True)

# Flatten columns if MultiIndex
spy.columns = [col[0].lower() if isinstance(col, tuple) else col.lower() 
               for col in spy.columns]

spy = spy[['close']].rename(columns={'close': 'market_close'}).reset_index()
spy.columns = ['date', 'market_close']
spy['date'] = pd.to_datetime(spy['date']).dt.date

# Step 2: Calculate daily market return
spy['market_return'] = spy['market_close'].pct_change()

# Step 3: Merge into your main dataframe
df_long['date'] = pd.to_datetime(df_long['date'])
spy['date'] = pd.to_datetime(spy['date'])

df_long = df_long.merge(spy[['date', 'market_return']], on='date', how='left')

# Step 4: Calculate abnormal return
# Abnormal return = what the stock actually did minus what the market did
df_long['abnormal_return'] = df_long['daily_return'] - df_long['market_return']

# Step 5: Sanity check — Jan 28 2021 should show massive abnormal returns
print(df_long[df_long['date'] == '2021-01-28']
      [['date', 'ticker', 'daily_return', 'market_return', 'abnormal_return']])


[*********************100%***********************]  1 of 1 completed

           date ticker  daily_return  market_return  abnormal_return
522  2021-01-28    AMC     -0.566332       0.009761        -0.576092
1653 2021-01-28     BB     -0.416335       0.009761        -0.426095
2784 2021-01-28   BBBY     -0.074087       0.009761        -0.083848
3915 2021-01-28    GME     -0.442894       0.009761        -0.452654
5046 2021-01-28    NOK     -0.283969       0.009761        -0.293730


## Check missing value 

In [25]:
# Overall missing values
print("=== Missing values per column ===")
print(df_long.isnull().sum())

# Per ticker — more informative
print("\n=== Missing values per ticker ===")
print(df_long.groupby('ticker')[['open','high','low','close','adj_close',
                                  'volume','daily_return','abnormal_return']]
      .apply(lambda x: x.isnull().sum()))

# Check BBBY specifically — where does its data end?
print("\n=== BBBY last 5 rows ===")
print(df_long[df_long['ticker']=='BBBY'].tail())

=== Missing values per column ===
date               0
ticker             0
open               0
high               0
low                0
close              0
adj_close          0
volume             0
daily_return       5
volume_mean        0
volume_std         0
return_mean        0
return_std         0
volume_zscore      0
return_zscore      5
volume_spike       0
return_spike       0
market_return      5
abnormal_return    5
dtype: int64

=== Missing values per ticker ===
        open  high  low  close  adj_close  volume  daily_return  \
ticker                                                            
AMC        0     0    0      0          0       0             1   
BB         0     0    0      0          0       0             1   
BBBY       0     0    0      0          0       0             1   
GME        0     0    0      0          0       0             1   
NOK        0     0    0      0          0       0             1   

        abnormal_return  
ticker                 

## Export data

In [26]:
import os

os.makedirs('data/processed', exist_ok=True)

# Drop the helper columns used for spike calculation — they're not needed in the final file
cols_to_drop = ['volume_mean', 'volume_std', 'return_mean', 'return_std']
df_final = df_long.drop(columns=cols_to_drop)

# Export
df_final.to_csv('data/processed/market_daily.csv', index=False)

# Confirm
print(f"Exported {len(df_final)} rows, {df_final['ticker'].nunique()} tickers")
print(f"Columns: {list(df_final.columns)}")
print(f"Date range: {df_final['date'].min()} to {df_final['date'].max()}")


Exported 5655 rows, 5 tickers
Columns: ['date', 'ticker', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'daily_return', 'volume_zscore', 'return_zscore', 'volume_spike', 'return_spike', 'market_return', 'abnormal_return']
Date range: 2019-01-02 00:00:00 to 2023-06-29 00:00:00


### Note:
daily_return, abnormal_return, return_zscore, and market_return are NaN for the first trading day of each ticker (2019-01-02) as no prior day exists for percentage change calculation. These 5 rows are retained in the dataset.